# Text-to-SQL Inventory Specialist - Fine-Tuning

Fine-tune Gemma 4 E2B with Unsloth + TRL SFT on inventory Text-to-SQL data.

Upload `data/dataset.jsonl` to Colab or mount from Google Drive before running.

In [ ]:
# Install Unsloth and all required dependencies for fine-tuning
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets python-dotenv huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-8_j5wnjj/unsloth_fe0a823b6fdc4066aa7e5521f5d2ce3e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-8_j5wnjj/unsloth_fe0a823b6fdc4066aa7e5521f5d2ce3e
  Resolved https://github.com/unslothai/unsloth.git to commit 5211b506e1d59d647528631e03cef1cb43ccbf7d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 138.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 26.5 MB/s eta 0:00:00
  

In [ ]:
# Import all required libraries and log in to Hugging Face
import json
import os
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

login(token="your-hugging-face-token")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Load the training dataset and format each example as instruction/output pairs
DATASET_PATH = "data/dataset.jsonl"

records = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

def format_example(example):
    return {"text": f"{example['instruction']}\n{example['output']}"}

dataset = Dataset.from_list([format_example(r) for r in records])
print(f"Loaded {len(dataset)} training examples")   b

Loaded 200 training examples


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-2B",
    max_seq_length=2048,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.6.9: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [ ]:
# Apply LoRA adapters to the model
# Only the small adapter layers are trained, not the full 4B parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [ ]:
# Fine-tune the model on our inventory SQL dataset using SFT Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=2,
        num_train_epochs=2,
        fp16=True,
        bf16=False,
        logging_steps=10,
        learning_rate=2e-4,
        dataset_text_field="text",
        max_seq_length=2048,
        report_to="none",
    ),
)

trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along 

Step,Training Loss
10,1.994785
20,0.555371
30,0.240468
40,0.216409
50,0.222428
60,0.149990
70,0.181594
80,0.169333
90,0.150240
100,0.144675


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.


TrainOutput(global_step=100, training_loss=0.4025293040275574, metrics={'train_runtime': 421.0169, 'train_samples_per_second': 0.95, 'train_steps_per_second': 0.238, 'total_flos': 294196671613440.0, 'train_loss': 0.4025293040275574, 'epoch': 2.0})

In [ ]:
# Push merged model to Hugging Face for local inference via Transformers
model.push_to_hub_merged(
    "Hecodes/text-to-sql-inventory",
    tokenizer=tokenizer,
    token="your-hugging-face-token",
    save_method="merged_16bit",
)
print("Model pushed to https://huggingface.co/Hecodes/text-to-sql-inventory")

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in Hecodes/text-to-sql-inventory/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00001.safetensors:   1%|          | 32.0MB / 4.55GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:10<00:00, 130.95s/it]


Unsloth: Merge process complete. Saved to `/content/Hecodes/text-to-sql-inventory`
Model pushed to https://huggingface.co/Hecodes/text-to-sql-inventory
